In [14]:
import requests
import json
import sqlite3
import numpy as np
import os

url  = ("https://raw.githubusercontent.com/AnalyseABO/"
        "Kart-fylker-og-kommuner-json/main/Bydeler_Oslo_u_marka.json")
r    = requests.get(url, timeout=15)
topo = r.json()

# ── INSPECT THE OBJECTS STRUCTURE ────────────────────────────────────────────

obj_name = list(topo["objects"].keys())[0]
print(f"Object name: {obj_name}")

obj      = topo["objects"][obj_name]
print(f"Object keys: {list(obj.keys())}")
print(f"Object type: {obj.get('type')}")

geometries = obj.get("geometries", [])
print(f"Geometries: {len(geometries)}")

if geometries:
    print(f"\nFirst geometry keys: {list(geometries[0].keys())}")
    print(f"First geometry properties: {geometries[0].get('properties', {})}")
    print(f"\nAll properties:")
    for g in geometries:
        print(f"  {g.get('properties', {})}")

Object name: Bydeler
Object keys: ['type', 'geometries']
Object type: GeometryCollection
Geometries: 16

First geometry keys: ['arcs', 'type', 'properties', 'id']
First geometry properties: {'kommunenum': '0301', 'BYDEL': '03', 'BYDELSNAVN': 'Sagene', 'Kombinert': '030103'}

All properties:
  {'kommunenum': '0301', 'BYDEL': '03', 'BYDELSNAVN': 'Sagene', 'Kombinert': '030103'}
  {'kommunenum': '0301', 'BYDEL': '10', 'BYDELSNAVN': 'Grorud', 'Kombinert': '030110'}
  {'kommunenum': '0301', 'BYDEL': '08', 'BYDELSNAVN': 'Nordre Aker', 'Kombinert': '030108'}
  {'kommunenum': '0301', 'BYDEL': '11', 'BYDELSNAVN': 'Stovner', 'Kombinert': '030111'}
  {'kommunenum': '0301', 'BYDEL': '14', 'BYDELSNAVN': 'Nordstrand', 'Kombinert': '030114'}
  {'kommunenum': '0301', 'BYDEL': '09', 'BYDELSNAVN': 'Bjerke', 'Kombinert': '030109'}
  {'kommunenum': '0301', 'BYDEL': '06', 'BYDELSNAVN': 'Ullern', 'Kombinert': '030106'}
  {'kommunenum': '0301', 'BYDEL': '05', 'BYDELSNAVN': 'Frogner', 'Kombinert': '030105'}
 

In [15]:
# ── MANUAL TOPOJSON → GEOJSON CONVERSION ─────────────────────────────────────

def decode_topojson(topo):
    """
    Manually convert TopoJSON to GeoJSON.
    Handles the arc decoding and coordinate transformation.
    """
    transform = topo.get("transform", {})
    scale     = transform.get("scale",     [1, 1])
    translate = transform.get("translate", [0, 0])
    arcs_raw  = topo["arcs"]

    # Decode delta-encoded arcs into absolute coordinates
    def decode_arc(arc):
        coords = []
        x, y   = 0, 0
        for dx, dy in arc:
            x += dx
            y += dy
            lon = x * scale[0] + translate[0]
            lat = y * scale[1] + translate[1]
            coords.append([lon, lat])
        return coords

    decoded_arcs = [decode_arc(arc) for arc in arcs_raw]

    def stitch_arcs(arc_indices):
        """Join arc segments into a ring"""
        ring = []
        for idx in arc_indices:
            if idx >= 0:
                arc = decoded_arcs[idx]
            else:
                arc = decoded_arcs[~idx][::-1]
            # Avoid duplicate junction points
            if ring:
                arc = arc[1:]
            ring.extend(arc)
        return ring

    def geometry_to_geojson(geom):
        gtype = geom["type"]
        if gtype == "Polygon":
            rings = [stitch_arcs(ring) for ring in geom["arcs"]]
            return {"type": "Polygon", "coordinates": rings}
        elif gtype == "MultiPolygon":
            polys = [
                [stitch_arcs(ring) for ring in poly]
                for poly in geom["arcs"]
            ]
            return {"type": "MultiPolygon", "coordinates": polys}
        return None

    # Build GeoJSON FeatureCollection
    obj_name   = list(topo["objects"].keys())[0]
    geometries = topo["objects"][obj_name]["geometries"]
    features   = []

    for geom in geometries:
        geo = geometry_to_geojson(geom)
        if geo:
            features.append({
                "type":       "Feature",
                "geometry":   geo,
                "properties": geom.get("properties", {})
            })

    return {"type": "FeatureCollection", "features": features}


geojson_dict = decode_topojson(topo)

print(f"✓ Converted: {len(geojson_dict['features'])} features")
print(f"\nProperty keys: {list(geojson_dict['features'][0]['properties'].keys())}")
print(f"\nAll bydel names:")
for feat in geojson_dict["features"]:
    props = feat["properties"]
    print(f"  {props}")

✓ Converted: 16 features

Property keys: ['kommunenum', 'BYDEL', 'BYDELSNAVN', 'Kombinert']

All bydel names:
  {'kommunenum': '0301', 'BYDEL': '03', 'BYDELSNAVN': 'Sagene', 'Kombinert': '030103'}
  {'kommunenum': '0301', 'BYDEL': '10', 'BYDELSNAVN': 'Grorud', 'Kombinert': '030110'}
  {'kommunenum': '0301', 'BYDEL': '08', 'BYDELSNAVN': 'Nordre Aker', 'Kombinert': '030108'}
  {'kommunenum': '0301', 'BYDEL': '11', 'BYDELSNAVN': 'Stovner', 'Kombinert': '030111'}
  {'kommunenum': '0301', 'BYDEL': '14', 'BYDELSNAVN': 'Nordstrand', 'Kombinert': '030114'}
  {'kommunenum': '0301', 'BYDEL': '09', 'BYDELSNAVN': 'Bjerke', 'Kombinert': '030109'}
  {'kommunenum': '0301', 'BYDEL': '06', 'BYDELSNAVN': 'Ullern', 'Kombinert': '030106'}
  {'kommunenum': '0301', 'BYDEL': '05', 'BYDELSNAVN': 'Frogner', 'Kombinert': '030105'}
  {'kommunenum': '0301', 'BYDEL': '01', 'BYDELSNAVN': 'Gamle Oslo', 'Kombinert': '030101'}
  {'kommunenum': '0301', 'BYDEL': '02', 'BYDELSNAVN': 'Grünerløkka', 'Kombinert': '030102'}


In [16]:
# ── SAVE TO DATABASE ──────────────────────────────────────────────────────────

BASE_DIR = os.path.dirname(os.path.abspath("app.py"))
DB_PATH  = os.path.join(BASE_DIR, "oslo_parcel.db")
conn     = sqlite3.connect(DB_PATH)

conn.execute("DROP TABLE IF EXISTS geojson")
conn.execute("CREATE TABLE geojson (id INTEGER PRIMARY KEY, data TEXT)")
conn.execute("INSERT INTO geojson (id, data) VALUES (1, ?)",
             (json.dumps(geojson_dict),))
conn.commit()
conn.close()

print(f"\n✓ Saved to database — restart Streamlit to see the choropleth map")


✓ Saved to database — restart Streamlit to see the choropleth map
